In [ ]:
import pandas as pd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data=pd.read_csv('PS_20174392719_1491204439457_log.csv')

isfraud-> the actual identifier whether its really fraud or nah
is flagged fraud->rule based flag

In [ ]:
display(data.head(3))
display(data.tail(3))
display(data.columns)
display(data.info())
display(data.dtypes)

In [ ]:
# Count the 0s (non-fraud) and 1s (fraud).
print(data["isFraud"].value_counts()) # the target

# Show the same breakdown as percentages.
print(data["isFraud"].value_counts(normalize=True) * 100)
#we see so much imbalance

In [ ]:
data["isFraud"].value_counts()
data["isFraud"].value_counts(normalize=True) * 100
sns.countplot(data=data, x="isFraud")

In [ ]:
money_columns = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
]

data[money_columns].describe().round(2)

In [ ]:
amount = data["amount"]
cutoff = amount.quantile(0.99)

# Zoom in on the lower 99% of amounts for this chart.
amount[amount <= cutoff].plot.hist(
    bins=30,
    title="Transaction amounts up to the 99th percentile",
    xlabel="Transaction amount",
)

In [ ]:
missing_count=data.isna().sum()
display(pd.DataFrame(
{
    "number of missing -> " : missing_count ,
    "PCT of missing -> ":(missing_count*100)/len(data)
}))

duplicate_count = data.duplicated().sum()
duplicate_pct=(duplicate_count*100)/len(data)


print(f"Exact duplicate rows beyond the first occurrence: {duplicate_count:,}")
print(f"Exact duplicate percentage: {duplicate_pct:.4f}%")

In [ ]:
train_eda = data.loc[data["step"] < 323]
fraud_by_type = train_eda.groupby("type").agg(
    total_transactions=("isFraud", "size"),#no of rows so all,
    fraud_count=("isFraud", "sum") #sum of 1 so sum of fraud,
)

fraud_by_type["fraud_rate_pct"] = (
    100 * fraud_by_type["fraud_count"] / fraud_by_type["total_transactions"]
)

display(fraud_by_type.sort_values("fraud_rate_pct", ascending=False).round(4))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plot_data = (
    fraud_by_type.sort_values("fraud_rate_pct", ascending=False)
    .reset_index()
)

plt.figure(figsize=(8, 4))
sns.barplot(data=plot_data, x="fraud_rate_pct", y="type")
plt.xlabel("Fraud rate (%)")
plt.ylabel("Transaction type")
plt.tight_layout()
plt.show()

In [ ]:
correlation_with_fraud = train_eda.corr(numeric_only=True)[["isFraud"]]
correlation_with_fraud = correlation_with_fraud.drop(index="isFraud")

sns.heatmap(correlation_with_fraud, annot=True, fmt=".3f",
            cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Correlation with fraud in the training period")
plt.show()

In [ ]:
sns.histplot(
    data=data,
    x="amount",
    hue="isFraud",
    stat="density",
    common_norm=False,
    log_scale=True,#due to therees range of millions in transactions,
    element="step"
)

In [ ]:
Q1 = data["amount"].quantile(0.25)
Q3 = data["amount"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = data[
    (data["amount"] < lower) |
    (data["amount"] > upper)

]

print(outliers)

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=data,
    x="isFraud",
    y="amount",
    showfliers=False
)

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=data,
    x="isFraud",
    y="amount"
)

plt.yscale("symlog")
plt.show()

In [ ]:
print("Total transactions:", len(data))
print("Total outliers:", len(outliers))
print("Outlier percentage:", len(outliers) / len(data) * 100)

remove old or new balance org and new balance dest bec 1 sns

We use the sender’s and recipient’s opening balances because they are available when a transaction is submitted. We exclude the balances recorded after the transaction so the demo can make a prediction using information available at that time.

In [ ]:
plt.figure(figsize=(12, 8))

sns.heatmap(
    data.corr(numeric_only=True),
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    vmin=-1,
    vmax=1,
    center=0
)

plt.title("Correlation Matrix")
plt.show()

In [ ]:
data["day"] = (data["step"] - 1) // 24 + 1

data["hour"] = (data["step"] - 1) % 24

train_eda = data.loc[data["step"] < 323]

In [ ]:
daily_fraud = data.groupby("day")["isFraud"].mean() * 100

plt.figure(figsize=(12, 5))

sns.lineplot(
    x=daily_fraud.index,
    y=daily_fraud.values,
    marker="o"
)

plt.xlabel("Day")
plt.ylabel("Fraud Rate (%)")
plt.title("Fraud Rate by Day")
plt.show()

In [ ]:
data.groupby("hour")["isFraud"].agg(
    ["count", "sum", "mean"]
)

In [ ]:
data.corr(numeric_only=True)["isFraud"].sort_values(
    ascending=False
)

In [ ]:
data["balance_ratio"] = (
    data["amount"] / data["oldbalanceOrg"].replace(0, np.nan)
)#8000 for millionaire is not like for brokie

In [ ]:
data["empties_account"] = (
    (data["oldbalanceOrg"] > 0) &
    np.isclose(
        data["amount"],
        data["oldbalanceOrg"],
        atol=0.01,
        rtol=0
    )
).astype(int)

In [ ]:
data.groupby("empties_account")["isFraud"].agg(
    ["count", "sum", "mean"]
)

In [ ]:
sns.barplot(
    data=data,
    x="empties_account",
    y="isFraud",
    errorbar=None
)

plt.ylabel("Fraud proportion")
plt.show()

In [ ]:
pd.crosstab(
    data["empties_account"],
    data["isFraud"]
)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    data["isFraud"],
    data["isFlaggedFraud"]
)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Existing Fraud Detection Rule")
plt.show()

In [ ]:
cm = confusion_matrix(
    data["isFraud"],
    data["isFlaggedFraud"]
)
#predicted is is flagged fraud so we compare actual vs predicted
cm

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    data["isFraud"],
    data["isFlaggedFraud"],
    digits=4
))

In [ ]:
original_features = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "oldbalanceDest"
]

engineered_features = [
    "day",
    "hour",
    "balance_ratio",
    "empties_account"
]

X_original = data[original_features]

X_engineered = data[
    original_features + engineered_features
]

y = data["isFraud"]

finally eda done
now preprocessing

In [ ]:
data = data.sort_values("step").reset_index(drop=True)

In [ ]:
X = data.drop(columns=[
    "isFraud",
    "isFlaggedFraud",
    "nameOrig",
    "nameDest",
    "newbalanceOrig",
    "newbalanceDest"
])

y = data["isFraud"]

In [ ]:
X_train = X[X["step"] < 323].copy()

X_val = X[
    (X["step"] >= 323) &
    (X["step"] < 378)
].copy()

X_test = X[X["step"] >= 378].copy()

y_train = y.loc[X_train.index]
y_val = y.loc[X_val.index]
y_test = y.loc[X_test.index]

In [ ]:
print(y_train.value_counts())
print(y_val.value_counts())
print(y_test.value_counts())

In [ ]:
steps = data["step"].unique()

print(steps)
print(len(steps))

In [ ]:
print(X_train["step"].max(), X_val["step"].min())
print(X_val["step"].max(), X_test["step"].min())
#chronic is correct

display(X_train.isnull().sum())
display(X_train.isnull().mean() * 100)

In [ ]:
X_train["balance_ratio"].isna().equals(
    X_train["oldbalanceOrg"] == 0
)

In [ ]:
for df in [X_train, X_val, X_test]:

    df["zero_balance"] = (
        df["oldbalanceOrg"] == 0
    ).astype(int)

    df["balance_ratio"] = df["balance_ratio"].fillna(0)

In [ ]:
X_train.isnull().sum()

In [ ]:
X_train.head()

In [ ]:
X_train.select_dtypes(include=["object", "category"]).columns

In [ ]:
X_train["type"].value_counts()

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore"
)

In [ ]:
encoded=encoder.fit_transform(X_train[["type"]])

In [ ]:
encoder.categories_

In [ ]:
encoder.get_feature_names_out()

In [ ]:
display(encoded[:5].toarray())
print(
    encoder.transform(
        X_train[["type"]].head()
    ).toarray()
)

In [ ]:
numerical_features = [
    "step",
    "amount",
    "oldbalanceOrg", 
    "oldbalanceDest",
    "day",
    "hour",
    "balance_ratio"
]

categorical_features = ["type"]

binary_features = [
    "empties_account",
    "zero_balance"
]

In [ ]:
pd.crosstab(y_val, X_val["empties_account"])

In [ ]:
features = [#not all can have outliers
    "amount",
    "oldbalanceOrg",
    "oldbalanceDest",
    "balance_ratio"
]

In [ ]:
results = []
any_outlier = pd.Series(False, index=X_train.index)

for feature in features:
    values = X_train[feature]

    # The ratio is undefined when the sender's opening balance is zero.
    # Exclude those placeholder zeros only when checking balance_ratio.
    if feature == "balance_ratio":
        values = values.where(X_train["zero_balance"] == 0)

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr

    below = values < lower_limit#wont happen mostly
    above = values > upper_limit
    flagged = below | above

    # Count each transaction once, even if several features flag it.
    any_outlier = any_outlier | flagged

    results.append({
        "feature": feature,
        "lower_limit": lower_limit,
        "upper_limit": upper_limit,
        "below_limit": below.sum(),#1 or 0 so number of rows appear
        "above_limit": above.sum(),
        "fraud_cases_flagged": y_train.loc[flagged].sum()
    })

outlier_summary = pd.DataFrame(results).set_index("feature")
display(outlier_summary.round(2))

print(f"Unique transactions flagged: {any_outlier.sum():,}")
print(f"Fraud cases flagged: {y_train.loc[any_outlier].sum():,} of {y_train.sum():,}")

so we cant remove outliers cuz most of features are outliers fraud so will be false if handled by removing

The IQR check flagged 1,672,817 training transactions. These included 2,215 of the 3,633 training frauds. Dropping every flagged row would have removed about 61% of our fraud examples. We kept all rows and used log1p on the transaction amount, both opening balances, and the balance ratio to compress large values.

lower bound doesnt mean this is in datset it means its the starting of the range

In [ ]:
X_train[features].min()

In [ ]:
X_train[features].skew()

In [ ]:
log_features = np.log1p(X_train[features])

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 16))

for i, col in enumerate(features):

    sns.histplot(
        X_train[col],
        bins=50,
        ax=axes[i, 0]
    )

    sns.histplot(
        log_features[col],
        bins=50,
        ax=axes[i, 1]
    )

    axes[i, 0].set_title(f"{col} - Before Log")
    axes[i, 1].set_title(f"{col} - After Log")

plt.tight_layout()
plt.show()

In [ ]:
numerical_features = [
    "step",
    "amount",
    "oldbalanceOrg",
    "oldbalanceDest",
    "day",
    "hour",
    "balance_ratio"
]

categorical_features = ["type"]

binary_features = ["empties_account", "zero_balance"]

In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

money_features = ["amount", "oldbalanceOrg", "oldbalanceDest", "balance_ratio"]
time_features = ["step", "day", "hour"]

money_pipeline = make_pipeline(
    FunctionTransformer(np.log1p),
    StandardScaler()
)

preprocessor = ColumnTransformer(
    transformers=[
        ("money", money_pipeline, money_features),
        ("time", StandardScaler(), time_features),
        ("category", OneHotEncoder(handle_unknown="ignore"), ["type"]),
        ("binary", "passthrough", ["empties_account", "zero_balance"])
    ]
)

In [ ]:
from sklearn.linear_model import LogisticRegression

baseline_model = make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=1000)
)

baseline_model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import confusion_matrix

# Predict fraud (1) or genuine (0) for each validation transaction.
model_predictions = baseline_model.predict(X_val)

# Use your existing rule as a second set of predictions.
rule_predictions = X_val["empties_account"]
existing_rule_predictions = data.loc[X_val.index, "isFlaggedFraud"]

results = []

for name, predictions in [
    ("Dataset rule (isFlaggedFraud)", existing_rule_predictions),
    ("Empties-account rule", rule_predictions),
    ("Logistic Regression", model_predictions)
]:
    tn, fp, fn, tp = confusion_matrix(
        y_val, predictions, labels=[0, 1]
    ).ravel()

    results.append({
        "method": name,
        "frauds_caught": tp,
        "frauds_missed": fn,
        "false_alarms": fp,
        "genuine_correctly_ignored": tn
    })

display(pd.DataFrame(results).set_index("method"))

the datasets rule based is very bad

On validation, the dataset’s isFlaggedFraud rule raised no alerts and missed all 570 frauds. The empties_account rule caught 565 of the 570 frauds with zero false alarms. We kept both as reference points for evaluating the learned models.

In [ ]:
fraud_only = pd.DataFrame({
    "actual": y_val,
    "rule_alert": rule_predictions,
    "model_alert": model_predictions
})

fraud_only = fraud_only.loc[fraud_only["actual"] == 1]

display(pd.crosstab(
    fraud_only["rule_alert"],
    fraud_only["model_alert"],
    rownames=["Empties-account rule"],
    colnames=["Logistic Regression"]
))

In [ ]:
fraud_scores = baseline_model.predict_proba(X_val)[:, 1]

results = []

for threshold in [0.50, 0.30, 0.10, 0.05]:
    alerts = fraud_scores >= threshold
    tn, fp, fn, tp = confusion_matrix(y_val, alerts, labels=[0, 1]).ravel()

    results.append({
        "threshold": threshold,
        "frauds_caught": tp,
        "frauds_missed": fn,
        "false_alarms": fp
    })

display(pd.DataFrame(results))#0.10 is good but have false alarms and 0.05 is better at cathcing frauds but have alot more flase alarms 

In [ ]:
threshold_results = pd.DataFrame(results).copy()

tp = threshold_results["frauds_caught"]
fp = threshold_results["false_alarms"]
fn = threshold_results["frauds_missed"]

threshold_results["precision"] = tp / (tp + fp)
threshold_results["recall"] = tp / (tp + fn)
threshold_results["f1"] = 2 * tp / (2 * tp + fp + fn)

display(threshold_results.round(4))

In [ ]:
baseline_threshold = 0.05
baseline_val_alerts = (fraud_scores >= baseline_threshold).astype(int)
print(confusion_matrix(y_val, baseline_val_alerts, labels=[0, 1]))

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone
from sklearn.metrics import average_precision_score
tree_model = make_pipeline(clone(preprocessor), DecisionTreeClassifier(max_depth=10,random_state=42))
tree_model.fit(X_train, y_train)

In [ ]:
tree_val_scores = tree_model.predict_proba(X_val)[:, 1]


In [ ]:
print("Logistic Regression AP:", round(average_precision_score(y_val, fraud_scores), 4))
print("Decision Tree AP:", round(average_precision_score(y_val, tree_val_scores), 4))

In [ ]:
tree_cases = pd.DataFrame({"score": tree_val_scores, "fraud": y_val})
tree_cutoffs = tree_cases.groupby("score")["fraud"].agg(["sum", "count"]).sort_index(ascending=False)
display(pd.DataFrame({
    "frauds_caught": tree_cutoffs["sum"].cumsum(),
    "false_alarms": (tree_cutoffs["count"] - tree_cutoffs["sum"]).cumsum()
}))

In [ ]:
# Use the fitted tree to score the training transactions.
tree_train_scores = tree_model.predict_proba(X_train)[:, 1]

# Measure how well it ranked frauds in the data it learned from.
print("Training AP:", round(average_precision_score(y_train, tree_train_scores), 4))

# Compare with its performance on the later validation transactions.
print("Validation AP:", round(average_precision_score(y_val, tree_val_scores), 4))

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

# Pair each model's validation scores with its chosen alert cutoff.
for name, scores, cutoff in [
    ("Logistic Regression", fraud_scores, 0.05),
    ("Decision Tree", tree_val_scores, 0.50)
]:
    # Convert scores into alerts using that model's cutoff.
    alerts = scores >= cutoff

    # Show precision, recall, and F1. Label 1 is the fraud class.
    print(name)
    print(classification_report(y_val, alerts, labels=[1],
                                target_names=["Fraud"], digits=4))

    # Use the original scores to calculate both ranking metrics.
    print("ROC-AUC:", round(roc_auc_score(y_val, scores), 4))
    print("Average precision:", round(average_precision_score(y_val, scores), 4))

In [ ]:
# Keep the training hours in chronological order.
steps = np.sort(X_train["step"].unique())

# Store one result for each amount of training history.
curve_rows = []

for fraction in [0.25, 0.50, 1.00]:
    # Include complete hours from the beginning of the training period.
    last_step = steps[int(len(steps) * fraction) - 1]
    rows = X_train["step"] <= last_step

    if fraction == 1.00:
        # Reuse the scores you already calculated for the full training set.
        train_scores = tree_train_scores
        val_scores = tree_val_scores
    else:
        # Fit a fresh copy so your selected tree stays available.
        smaller_tree = clone(tree_model)
        smaller_tree.fit(X_train.loc[rows], y_train.loc[rows])

        # Score its own training rows and the same validation period each time.
        train_scores = smaller_tree.predict_proba(X_train.loc[rows])[:, 1]
        val_scores = smaller_tree.predict_proba(X_val)[:, 1]

    curve_rows.append({
        "training_rows": int(rows.sum()),
        "training_AP": average_precision_score(y_train.loc[rows], train_scores),
        "validation_AP": average_precision_score(y_val, val_scores)
    })

curve = pd.DataFrame(curve_rows)
display(curve.round(4))
curve.plot(x="training_rows", y=["training_AP", "validation_AP"], marker="o")
plt.xlabel("Number of training transactions")
plt.ylabel("Average precision")
plt.title("Decision Tree learning curve")
plt.show()

For the unweighted Decision Tree, validation average precision increased from 0.2244 with the shortest training history to 0.9930 with the full training history. Training average precision was 0.9974 on the full training set. Because each larger subset also includes later transactions, this experiment shows the benefit of using the full available history but cannot isolate the effect of sample size alone.

In [ ]:
weighted_tree = clone(tree_model)
weighted_tree.set_params(decisiontreeclassifier__class_weight="balanced")
weighted_tree.fit(X_train, y_train)

In [ ]:
# Get one fraud score per validation transaction.
weighted_scores = weighted_tree.predict_proba(X_val)[:, 1]

# Start with the same cutoff used for the unweighted tree.
weighted_alerts = weighted_scores >= 0.50

# Count correct and incorrect decisions against the actual labels.
tn, fp, fn, tp = confusion_matrix(y_val, weighted_alerts, labels=[0, 1]).ravel()

print("Frauds caught:", tp)
print("Frauds missed:", fn)
print("False alarms:", fp)
print("Average precision:", round(average_precision_score(y_val, weighted_scores), 4))

Limitations and next improvement. PaySim contains simulated transactions, so these results describe performance on this simulation. Some early exploration included labels from the full dataset before we revised the charts to use the training period. The weighted tree’s displayed percentage is a model score. Its calibration as a real-world fraud probability has not been tested. A useful next evaluation would use a new time period and separately measure recall on transfers that leave a positive sender balance, since all three missed frauds shared that pattern.

In [ ]:
for cutoff in [0.50, 0.75, 0.90, 0.99]:
    alerts = weighted_scores >= cutoff
    tn, fp, fn, tp = confusion_matrix(y_val, alerts, labels=[0, 1]).ravel()
    print(f"Cutoff {cutoff:.2f} | Caught {tp} | Missed {fn} | False alarms {fp}")

### Why we chose these thresholds

For Logistic Regression, lowering the threshold from 0.10 to 0.05 caught four more frauds on the validation set (563 → 567). It also flagged four more genuine transactions (1 → 5 false alarms). We chose 0.05 because we considered four extra reviews acceptable to catch four additional frauds.

For the weighted Decision Tree, all four tested thresholds caught all 570 validation frauds and raised one false alarm. We kept 0.50 because raising the threshold gave no improvement on validation. We then used these chosen thresholds for the final test evaluation.

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

final_model = weighted_tree
final_threshold = 0.50

for name, model, cutoff in [
    ("Baseline: Logistic Regression", baseline_model, 0.05),
    ("Final: Weighted Decision Tree", final_model, final_threshold)
]:
    scores = model.predict_proba(X_test)[:, 1]
    alerts = scores >= cutoff
    tn, fp, fn, tp = confusion_matrix(y_test, alerts, labels=[0, 1]).ravel()

    print(f"\n{name} | threshold {cutoff:.2f}")
    print(f"Frauds caught: {tp} | Frauds missed: {fn} | False alarms: {fp}")
    print(classification_report(y_test, alerts, labels=[1],
                                target_names=["Fraud"], digits=4, zero_division=0))
    print(f"ROC-AUC: {roc_auc_score(y_test, scores):.4f}")
    print(f"Average precision: {average_precision_score(y_test, scores):.4f}")

In [ ]:
# Get the final tree's fraud score for every test transaction.
tree_test_scores = final_model.predict_proba(X_test)[:, 1]

# Start a table with the transaction details and the actual fraud label.
review = X_test[["type", "amount"]].copy()
review["isFraud"] = y_test

# Apply the saved threshold to get the tree's prediction for each row.
review["prediction"] = (tree_test_scores >= final_threshold).astype(int)
review["fraud_score"] = tree_test_scores

# Show the rows where the prediction differs from the actual label.
review[review["prediction"] != review["isFraud"]]

In [ ]:
mistakes = review["prediction"] != review["isFraud"]
X_test.loc[mistakes, ["type", "amount", "oldbalanceOrg", "empties_account"]]

On the PaySim test set, the weighted Decision Tree detected 4,007 of 4,010 frauds and raised one false alarm. All three missed frauds were transfers whose amounts were exactly 10,000,000 below the sender’s opening balance. The false alarm was a genuine cash-out of 202,978.56 from a recorded opening balance of zero.

In [ ]:
import joblib

joblib.dump(
    {"model": final_model, "threshold": final_threshold},
    "fraud_detector.joblib"
)

In [ ]:
# Load the model and threshold you saved.
saved = joblib.load("fraud_detector.joblib")

# Select one transaction you already reviewed, keeping it as a one-row table.
transaction = X_test.loc[[6323736]]

# Ask the saved model for its fraud score.
score = saved["model"].predict_proba(transaction)[0, 1]

# Turn the score into the demo label using the saved threshold.
print("Suspicious" if score >= saved["threshold"] else "Normal")
print("Fraud score:", score)

In [ ]:
# Start with only the five values the demo user would provide.
raw_columns = ["step", "type", "amount", "oldbalanceOrg", "oldbalanceDest"]
demo_input = transaction[raw_columns].copy()

# Convert the simulation step into day and hour. Step 1 is day 1, hour 0.
demo_input["day"] = (demo_input["step"] - 1) // 24 + 1
demo_input["hour"] = (demo_input["step"] - 1) % 24

# Record whether the sender started with a zero balance.
balance = demo_input["oldbalanceOrg"]
demo_input["zero_balance"] = (balance == 0).astype(int)

# Calculate amount / opening balance. Use the training placeholder for division by zero.
demo_input["balance_ratio"] = (demo_input["amount"] / balance.replace(0, np.nan)).fillna(0)

# Flag amounts that match a positive opening balance within one cent.
demo_input["empties_account"] = (
    (balance > 0) & np.isclose(demo_input["amount"], balance, atol=0.01, rtol=0)
).astype(int)

# Compare every rebuilt input with the original row used by the model.
print(demo_input[transaction.columns].equals(transaction))

In [ ]:
# Get the fraud score from the inputs you just rebuilt.
demo_score = saved["model"].predict_proba(demo_input)[0, 1]

# Use the saved threshold to display the decision.
print("Suspicious" if demo_score >= saved["threshold"] else "Normal")

# Check that the score matches the original prediction for this row.
print("Same score as before:", demo_score == score)

In [ ]:
# Get the final tree's score for each test transaction.
scores = final_model.predict_proba(X_test)[:, 1]

# Find scores that display strictly between 0.00% and 100.00% in your app.
percent = (scores * 100).round(2)
middle = (percent > 0) & (percent < 100)

# Show one matching transaction using the five fields your app asks for.
fields = ["step", "type", "amount", "oldbalanceOrg", "oldbalanceDest"]
example = X_test.loc[middle, fields].copy()
example["score_percent"] = percent[middle]
display(example.head(1))